# Session Behavioral Fingerprinting — EDA

Exploration only. Any logic worth keeping graduates into `features.py` / `charts.py` as a named function — this notebook should only ever call into those modules, not reimplement analysis logic inline.

**Setup** (see `../README.md` for the full version): run once per machine —
```
uv sync --group test --group analysis
uv run python -m ipykernel install --user --name bear-trap-analysis --display-name "Python (bear-trap analysis)" --env PYTHONPATH "$(pwd)"
```
then select the "Python (bear-trap analysis)" kernel for this notebook — the same one `network/notebooks/eda.ipynb` uses. It's pinned to this repo's `.venv` with the repo root baked onto PYTHONPATH, so `honeypot.analysis.*` absolute imports resolve directly below, deps always match `uv.lock`, and there's no per-submodule scoping needed. This notebook assumes that kernel is selected — if imports fail, check the kernel picker rather than expecting a `sys.path` fallback.

See `.claude/tasks/active-task.md` for the full spec (local/gitignored — won't exist on a fresh clone).

In [ ]:
import polars as pl

from honeypot.analysis.common.extract import get_sessions
from honeypot.analysis.fingerprinting.charts import (
    _cluster_command_sequences,
    _cluster_telnet_options,
    plot_client_fingerprint_breakdown,
    plot_command_sequence_clusters,
    plot_inter_command_timing,
    plot_telnet_option_breakdown,
)
from honeypot.analysis.fingerprinting.features import extract_session_features

## Load data

`"offline"` reads the most recent local snapshot (no GCP credentials
needed). Switch to `"live"` to pull fresh from BigQuery and refresh the
snapshot as a side effect.

In [ ]:
MODE = "offline"

sessions = get_sessions(MODE)
sessions.head()

In [ ]:
features = extract_session_features(sessions)
features.describe()

## Baseline charts

`charts.py` forces the `Agg` backend (headless-safe for tests/`pipeline.py`), so these calls save a PNG but won't render inline on their own — wrap with `Image(...)` to preview.

In [ ]:
from IPython.display import Image

Image(plot_client_fingerprint_breakdown(features))

In [ ]:
Image(plot_telnet_option_breakdown(features))

In [ ]:
Image(plot_inter_command_timing(features))

In [ ]:
Image(plot_command_sequence_clusters(features))

## Exploration

Ad hoc from here — human vs. bot vs. agent timing separation, HASSH breakdown by fingerprint bucket, command-sequence patterns, whatever the data suggests.

In [ ]:
features.filter(pl.col("client_version").is_not_null())

In [ ]:
sessions.filter(pl.col("agent_type").is_not_null())

### Command-sequence clusters

`command_sequence_hash` is normalized before hashing (busybox verification
tags stripped, embedded C2 IPs masked, consecutive-repeat commands
collapsed) so near-identical playbook variants group together -- see
`features._hash_commands`. Singletons (count == 1) have no match to a
known playbook and are the most interesting sessions to review by hand.

In [ ]:
clusters = _cluster_command_sequences(features)
singletons = clusters.filter(pl.col("count") == 1)

print(f"{clusters.height} distinct command sequences")
print(f"{singletons.height} singletons (no match to a known playbook)")
print(f"{int(clusters['count'].sum())} sessions ran at least one command")

clusters.head(10)

In [ ]:
# Full detail on the singleton sessions -- src_ip/client/timing context
# plus the actual commands, for manual review.
singleton_hashes = singletons["command_sequence_hash"].to_list()
features.filter(pl.col("command_sequence_hash").is_in(singleton_hashes)).select(
    "session",
    "src_ip",
    "client_version",
    "is_ssh_client",
    "command_count",
    "inter_command_mean_s",
    "has_agent_report",
    "commands",
)

### Telnet option-sequence clusters

Telnet has no version banner/kex exchange the way SSH does, so
`telnet_option_hash` (the ordered IAC command + option-name sequence from
`cowrie.telnet.option`, see `features._hash_telnet_options`) is the closest
per-client fingerprint available. Singletons here are telnet sessions with
no match to a known client's negotiation pattern.

In [ ]:
telnet_clusters = _cluster_telnet_options(features)
telnet_singletons = telnet_clusters.filter(pl.col("count") == 1)

print(f"{telnet_clusters.height} distinct telnet option sequences")
print(f"{telnet_singletons.height} singletons (no match to a known client)")
print(
    f"{int(telnet_clusters['count'].sum())} telnet sessions negotiated "
    "at least one option"
)

telnet_clusters.head(10)

In [ ]:
# Full detail on the singleton telnet sessions, plus the exploit-attempt
# flag Cowrie already parses for us (see cowrie.telnet.exploit_attempt).
telnet_singleton_hashes = telnet_singletons["telnet_option_hash"].to_list()
features.filter(pl.col("telnet_option_hash").is_in(telnet_singleton_hashes)).select(
    "session",
    "src_ip",
    "telnet_options",
    "command_count",
    "inter_command_mean_s",
    "has_exploit_attempt",
    "exploit_cve",
    "has_agent_report",
    "commands",
)

### Telnet sessions with no option-negotiation data

Not every telnet session emits a `cowrie.telnet.option` event -- these get
`telnet_option_hash is None` and are invisible to the clustering above.
First live pull (2026-08-22 snapshot) showed roughly 28% of telnet
sessions fell in this bucket. Worth understanding whether that's scanners
disconnecting before negotiation, a Cowrie/Vector logging gap, or its own
distinct population.

In [ ]:
telnet = features.filter(pl.col("protocol") == "telnet")
no_options = telnet.filter(pl.col("telnet_option_hash").is_null())

print(f"{no_options.height} of {telnet.height} telnet sessions have no option data")
print("event_count distribution for this residual:")
print(no_options["event_count"].describe())

no_options.select(
    "session",
    "src_ip",
    "event_count",
    "command_count",
    "has_exploit_attempt",
    "commands",
).head(20)